# H&M M4 귀속 3시드 — 중단된 seed 43·44 재개

9/21에 사전등록하고 실행하다 중단한 실험을 **중단 지점부터 이어서** 돌립니다. 새 설계가 아닙니다.

**무엇을 보는 실험인가.** 한 seed당 세 arm을 같은 조건으로 학습합니다.

| arm | 내용 |
|---|---|
| M1 | 기준모형 |
| 실제 `q_C` M4 | `w(u,i) = [1 + 0.5·q_C(u)·상품가격백분위(i)·고객경제구간적합도(u,i)] / 평균` |
| 순열 `q_C` M4 | **`q_C`만** binary degree 10분위 안에서 섞음. 가격·적합도·그래프·음성·초기화는 고정 |

순열 arm이 있어 **"개선이 CLV 배정 때문인가, 그냥 가중치가 있어서인가"**를 분리할 수 있습니다.

**사전등록된 판정(코드에 박혀 있음).** 실제 M4가 (1) M1보다 두 경제지표 모두 우위, (2) 순열 M4보다 두 경제지표 모두 우위, (3) 여섯 정확도가 M1의 99% 이상, (4) Top-10이 실제로 바뀌고 행가중치 변동계수가 0보다 큼 — 네 조건을 모두 만족할 때만 통과입니다. seed 42는 통과했고, **3시드 중 2시드 이상 통과해야** 5시드로 확대합니다.

**왜 옛 커밋으로 고정하나.** 체크포인트 재개는 소스 커밋이 같아야 통과하고, 무엇보다 seed 42가 그 커밋으로 실행됐습니다. 최신 코드로 돌리면 seed 간 비교가 흔들립니다.

**중단 지점**: seed 43은 M1 epoch 65/100, seed 44는 M1 epoch 66/100. 실제·순열 arm은 아직 시작 전입니다.

**비용**: arm당 100 epoch 약 6.7시간. 이 노트북은 한 seed를 끝까지 돌립니다(M1 잔여 + 실제 + 순열 ≈ 16시간). **seed 43과 44를 서로 다른 Colab 런타임에서 동시에 돌리면** 벽시계로 절반입니다.

단일 개발분할이며 test/holdout은 열지 않습니다.

In [ ]:
SEED = 43        # 이 런타임에서 돌릴 시드. 다른 탭에서 44로 두고 동시에 돌리면 된다.

from pathlib import Path
import json, os, subprocess, sys
from google.colab import drive

# 9/21 중단 시점의 체크포인트가 기록한 값 — 재개가 성립하는지 여기서 검증한다
EXPECTED = {43: {'config_hash': '7218001fef1d', 'stopped_epoch': 65},
            44: {'config_hash': '429a3169f0d7', 'stopped_epoch': 66}}
SOURCE_COMMIT = '3188b01c360d54e78338eaebdf2a8ecb6a6d52d8'
if SEED not in EXPECTED:
    raise ValueError(f'사전등록된 후속 시드는 43·44뿐입니다: {SEED}')

ROOT = Path('/content/drive/MyDrive/논문/data')
OUT = ROOT/'results_v3_hm_m4_k1_assignment_control_hm2y_development_screen_v1'
if not OUT.is_dir():
    if os.path.ismount('/content/drive'):
        raise RuntimeError(f'Drive는 연결됐지만 기존 결과 폴더가 없습니다: {OUT}. 처음부터 학습하지 않는다.')
    try:
        drive.mount('/content/drive')
    except (ValueError, NotImplementedError) as exc:
        raise RuntimeError('Drive 연결 실패. 학습은 시작되지 않았습니다.') from exc
    if not OUT.is_dir():
        raise RuntimeError(f'기존 결과 폴더가 없습니다: {OUT}')

REPO = Path('/content/clv-m4-assign-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent == REPO.resolve(), \
        '다른 소스가 로드되어 있습니다. 런타임을 다시 시작하세요.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import lightgcn_clv_m4_k1_assignment_control_hm2y as m4
cfg = m4.configure_hm2y_m4_assignment_screen(seed=SEED)
print(json.dumps(m4.preflight_summary(cfg), ensure_ascii=False, indent=2))

## 1. 재개가 성립하는지 확인 (학습 없음)

중단된 체크포인트를 실제로 이어받는지 **학습을 시작하기 전에** 확인합니다. 설정 해시가 9/21과 다르면 같은 폴더를 쓰지 않아 **조용히 처음부터** 돌게 되므로, 그 경우 여기서 멈춥니다.

데이터 준비에 2분 정도 걸립니다.

In [ ]:
prepared = m4._prepare(cfg)
actual_hash = prepared['config_hash']
expected_hash = EXPECTED[SEED]['config_hash']
print(f'설정 해시: {actual_hash} (9/21 기록: {expected_hash})')
if actual_hash != expected_hash:
    raise RuntimeError(
        f'설정 해시가 9/21과 다릅니다({actual_hash} != {expected_hash}). '
        '이대로 돌리면 중단 지점을 못 이어받고 처음부터 학습합니다. 중단합니다.')

progress = OUT/'progress'/actual_hash
for name in sorted(p.name for p in (progress/'stages').glob('*.json')):
    state = json.loads((progress/'stages'/name).read_text())
    print(f"  {name}: {state.get('status')} | epoch {state.get('epoch')}/{state.get('max_epoch')}")
resume = progress/'resume'/f'hm2y_development_train_m1_bpr_k1_hm2y_m4_assignment_control_s{SEED}_latest.pt'
print('M1 재개 파일:', '있음' if resume.is_file() else '없음 — 처음부터 학습하게 됩니다')

## 2. 세 arm 학습 (약 16시간)

M1은 중단된 epoch 다음부터 이어지고, 실제·순열 arm은 처음부터 100 epoch을 돕니다. epoch마다 Drive에 저장하므로 끊기면 같은 셀을 다시 실행하면 이어집니다. 완료된 arm은 건너뜁니다.

In [ ]:
import torch
assert torch.cuda.is_available(), '전체 실험은 GPU 런타임을 사용하세요.'
frame = m4.run_hm2y_m4_assignment_screen(cfg)

## 3. 귀속 판정과 원본 ZIP

`attribution_pass`가 이 seed의 판정입니다. **한 seed만으로 결론내지 않습니다** — 사전등록대로 42·43·44 중 2시드 이상이 통과해야 다음 단계로 갑니다. 순열 대비 우위(`actual_beats_shuffle_on_both_economic_metrics`)가 CLV 배정에 귀속할 수 있는지를 가르는 항목입니다.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

reading = frame.attrs.get('reading', {})
print(json.dumps(reading, ensure_ascii=False, indent=2))
print()
print(frame.to_string(index=False))

paths = frame.attrs.get('result_paths', {})
zip_path = Path(f'/content/m4_assignment_hm2y_s{SEED}_results.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for path in paths.values():
        archive.write(path, arcname=Path(path).name)
files.download(str(zip_path))